## Desafio D — Sistema de Registro de Preços

### Problema

Quais características diferenciam contratações realizadas com e sem Sistema de Registro de Preços?

### Possíveis perguntas

- O SRP é mais frequente em determinados tipos de contratação?
- Existem diferenças nos valores das contratações?
- Determinados órgãos utilizam SRP proporcionalmente mais do que outros?

### Variável de interesse

Quando disponível:

```text
srp
```

### Possíveis análises

- proporções;
- tabelas cruzadas;
- comparação de valores;
- teste qui-quadrado.


documentação API: https://dadosabertos.compras.gov.br/swagger-ui/index.html


In [20]:
#!pip install requests pandas matplotlib -q

In [21]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [22]:
BASE_URL = "https://dadosabertos.compras.gov.br"

ENDPOINT_CONTRATACOES = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"

url = BASE_URL + ENDPOINT_CONTRATACOES

print(url)

https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133


In [23]:
def extrair_registros(json_resposta):
    if isinstance(json_resposta, list):
        return json_resposta

    if not isinstance(json_resposta, dict):
        return []

    for chave in ["resultado", "resultados", "data", "content"]:
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave]

    return []



# Pegando dados 2024

In [24]:
modalidades = [5, 6, 8, 9] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [25]:

for modalidade in modalidades:
    params = {
        "pagina": 1,
        "tamanhoPagina": 100,
        "dataPublicacaoPncpInicial": "2024-01-01",
        "dataPublicacaoPncpFinal": "2024-12-31",
        "codigoModalidade": modalidade
    }

    resposta = requests.get(
        url,
        params=params,
        timeout=60
        )

    if resposta.status_code == 200:
        dados = resposta.json()
        registros = extrair_registros(dados)
        todos_registros.extend(registros)
        print(
            f"Modalidade {modalidade}: {len(registros)} registros coletados."
        )



Modalidade 5: 100 registros coletados.
Modalidade 6: 100 registros coletados.
Modalidade 8: 0 registros coletados.
Modalidade 9: 0 registros coletados.


In [26]:
# print("JSON retornado:", registros)
df_24 = pd.json_normalize(todos_registros)
df_24.head()

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,None,45178,MINISTERIO DA FAZENDA,None,F,...,Edital,Aberto-Fechado,614963.83,343793.5,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False
1,41300105000262023,02030715000112-1-000226/2023,2023,226,02030715000112,None,89804,AGENCIA NACIONAL DE TELECOMUNICACOES,None,F,...,Edital,Aberto-Fechado,404.74,NaN,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T08:00:00,2024-01-17T10:00:00,False
2,15812605000492023,10729992000146-1-000127/2023,2023,127,10729992000146,None,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",None,F,...,Edital,Aberto-Fechado,105000.00,105000.0,2024-01-02T07:00:05,2024-01-15T07:07:29,2024-01-02T07:00:05,2024-01-02T08:00:00,2024-01-16T10:00:00,False
3,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,None,43849,COMANDO DA AERONAUTICA,None,F,...,Edital,Aberto,286011.82,270000.0,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False
4,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,None,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,None,F,...,Edital,Aberto,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False


In [27]:
#for i in todos_registros:
    #print(i)

In [28]:

# 2. PROCURA AUTOMÁTICA: Procura qualquer coluna que tenha 'srp' ou 'preco' no nome
colunas_srp = [
    c
    for c in df_24.columns
    if 'srp' in c.lower() or 'registropreco' in c.lower() or 'preco' in c.lower()
]
print("Colunas encontradas relacionadas a SRP/Preço:")
print(colunas_srp)

# Se encontrou alguma coluna compatível, renomeia a primeira para 'srp'
if colunas_srp:
    coluna_identificada = colunas_srp[0]
    print(f"\nUsando a coluna '{coluna_identificada}' como 'srp'")
    df_24 = df_24.rename(columns={coluna_identificada: 'srp'})
else:
    print(
        "\nNenhuma coluna de SRP foi encontrada diretamente. Veja todas as colunas:"
    )
    print(df_24.columns.tolist())

# 3. Mapeia outras colunas comuns do PNCP
mapeamento = {
    'orgaoEntidade.razaoSocial': 'orgaoEntidadeRazaoSocial',
    'modalidadeNome': 'modalidadeNome',
    'numeroCompra': 'numeroCompra',
    'objetoCompra': 'objetoCompra',
    'valorTotalEstimado': 'valorTotalEstimado',
    'valorTotalHomologado': 'valorTotalHomologado',
}
df_24 = df_24.rename(
    columns={k: v for k, v in mapeamento.items() if k in df_24.columns}
)

df_24["ano"] = 2024


# 4. Trata e padroniza a coluna SRP (identifica True, 1, 'True', 'S', etc.)
if 'srp' in df_24.columns:
    # Mostra os valores brutos que vieram da API antes de converter
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_24['srp'].value_counts(dropna=False))

    # Converte para booleano real
    df_24['srp_bool'] = df_24['srp'].astype(str).str.lower().isin(['true', '1', 's', 'sim'])

    df_24_com_srp = df_24[df_24['srp_bool'] == True]
    df_24_sem_srp = df_24[df_24['srp_bool'] == False]

    print(f"\n Total COM SRP: {len(df_24_com_srp)}")
    print(f" Total SEM SRP: {len(df_24_sem_srp)}")


Colunas encontradas relacionadas a SRP/Preço:
['srp']

Usando a coluna 'srp' como 'srp'

Valores brutos encontrados na coluna SRP:
srp
False    152
True      48
Name: count, dtype: int64

 Total COM SRP: 48
 Total SEM SRP: 152


In [29]:
print("\nDataFrame com SRP:")
display(df_24_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_24_sem_srp.head())


DataFrame com SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,None,45178,MINISTERIO DA FAZENDA,None,F,...,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2024,True
7,92804805900012024,88830609000139-1-000001/2024,2024,1,88830609000139,None,41243,MUNICIPIO DE CAXIAS DO SUL,None,M,...,68469.70,27300.60,2024-01-02T07:00:16,2024-01-02T07:00:16,2024-01-02T07:00:16,2024-01-02T08:00:00,2024-01-16T08:30:00,False,2024,True
9,15590105000962023,15126437000143-1-003231/2023,2023,3231,15126437000143,None,95159,EMPRESA BRASILEIRA DE SERVIÇOS HOSPITALARES,None,F,...,137987.47,103025.62,2024-01-02T07:00:20,2024-01-02T07:00:20,2024-01-02T07:00:20,2024-01-02T08:00:00,2024-01-12T09:00:00,False,2024,True
11,15812605000352023,10729992000146-1-000128/2023,2023,128,10729992000146,None,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",None,F,...,13225.77,1600.00,2024-01-02T07:00:23,2024-01-02T07:00:23,2024-01-02T07:00:23,2024-01-02T08:00:00,2024-01-22T10:00:00,False,2024,True
12,16039905000432023,00394452000103-1-014522/2023,2023,14522,00394452000103,None,44611,COMANDO DO EXERCITO,None,F,...,6389632.82,1145924.50,2024-01-02T07:00:24,2024-01-05T07:04:20,2024-01-02T07:00:24,2024-01-05T08:00:00,2024-01-17T09:00:00,False,2024,True



DataFrame sem SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
1,41300105000262023,02030715000112-1-000226/2023,2023,226,02030715000112,None,89804,AGENCIA NACIONAL DE TELECOMUNICACOES,None,F,...,404.74,NaN,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T08:00:00,2024-01-17T10:00:00,False,2024,False
2,15812605000492023,10729992000146-1-000127/2023,2023,127,10729992000146,None,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",None,F,...,105000.00,105000.0,2024-01-02T07:00:05,2024-01-15T07:07:29,2024-01-02T07:00:05,2024-01-02T08:00:00,2024-01-16T10:00:00,False,2024,False
3,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,None,43849,COMANDO DA AERONAUTICA,None,F,...,286011.82,270000.0,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2024,False
4,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,None,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,None,F,...,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2024,False
5,92668605000222023,04801221000110-1-000444/2023,2023,444,04801221000110,None,53800,TRIBUNAL DE CONTAS DO ESTADO DE RONDONIA,None,E,...,99975.00,95466.0,2024-01-02T07:00:10,2024-01-02T07:00:10,2024-01-02T07:00:10,2024-01-02T08:00:00,2024-01-16T10:00:00,False,2024,False


In [30]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_24 = df_24.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao_24.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao_24)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,2,77,152
Com SRP,1,25,48


# Pegando dados 2025

In [31]:
modalidades = [5, 6, 8, 9] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [32]:

for modalidade in modalidades:
    params = {
        "pagina": 1,
        "tamanhoPagina": 100,
        "dataPublicacaoPncpInicial": "2025-01-01",
        "dataPublicacaoPncpFinal": "2025-12-31",
        "codigoModalidade": modalidade
    }

    resposta = requests.get(
        url,
        params=params,
        timeout=60
        )

    if resposta.status_code == 200:
        dados = resposta.json()
        registros = extrair_registros(dados)
        todos_registros.extend(registros)
        print(
            f"Modalidade {modalidade}: {len(registros)} registros coletados."
        )



Modalidade 5: 100 registros coletados.
Modalidade 6: 100 registros coletados.
Modalidade 8: 0 registros coletados.
Modalidade 9: 0 registros coletados.


In [33]:
# print("JSON retornado:", registros)
df_25 = pd.json_normalize(todos_registros)
df_25.head()

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,92647305900242024,06272868000127-1-000056/2024,2024,56,06272868000127,None,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,None,F,...,Edital,Aberto-Fechado,400000.00,NaN,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T08:00:00,2025-01-16T09:00:00,False
1,92647305900252024,06272868000127-1-000057/2024,2024,57,06272868000127,None,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,None,F,...,Edital,Aberto-Fechado,250782.10,151826.000,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T08:00:00,2025-01-14T09:00:00,False
2,38910305900162024,21947619000188-1-000068/2024,2024,68,21947619000188,None,21599,CONSELHO REGIONAL DE FISIOTERAPIA E TERAPIA OC...,None,F,...,Edital,Aberto-Fechado,868719.38,NaN,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:00:00,2025-01-17T09:00:00,False
3,94300105913862024,07954480000179-1-023839/2024,2024,23839,07954480000179,None,57870,ESTADO DO CEARA,None,E,...,Edital,Aberto-Fechado,11147211.58,6619438.535,2025-01-02T09:03:47,2025-05-09T08:13:51,2025-01-02T09:03:47,2025-04-25T08:00:00,2025-05-15T09:00:00,False
4,38918505900112024,00119784000171-1-000036/2024,2024,36,00119784000171,None,43167,CONSELHO FEDERAL DE MEDICINA VETERINARIA,None,F,...,Edital,Aberto,4612985.04,3718205.520,2025-01-02T09:03:51,2025-09-22T07:36:06,2025-01-02T09:03:51,2025-09-22T08:00:00,2025-10-07T10:00:00,False


In [34]:

# 2. PROCURA AUTOMÁTICA: Procura qualquer coluna que tenha 'srp' ou 'preco' no nome
colunas_srp = [
    c
    for c in df_25.columns
    if 'srp' in c.lower() or 'registropreco' in c.lower() or 'preco' in c.lower()
]
print("Colunas encontradas relacionadas a SRP/Preço:")
print(colunas_srp)

# Se encontrou alguma coluna compatível, renomeia a primeira para 'srp'
if colunas_srp:
    coluna_identificada = colunas_srp[0]
    print(f"\nUsando a coluna '{coluna_identificada}' como 'srp'")
    df_25 = df_25.rename(columns={coluna_identificada: 'srp'})
else:
    print(
        "\nNenhuma coluna de SRP foi encontrada diretamente. Veja todas as colunas:"
    )
    print(df_25.columns.tolist())

# 3. Mapeia outras colunas comuns do PNCP
mapeamento = {
    'orgaoEntidade.razaoSocial': 'orgaoEntidadeRazaoSocial',
    'modalidadeNome': 'modalidadeNome',
    'numeroCompra': 'numeroCompra',
    'objetoCompra': 'objetoCompra',
    'valorTotalEstimado': 'valorTotalEstimado',
    'valorTotalHomologado': 'valorTotalHomologado',
}
df_25 = df_25.rename(
    columns={k: v for k, v in mapeamento.items() if k in df_25.columns}
)

df_25["ano"] = 2025

# 4. Trata e padroniza a coluna SRP (identifica True, 1, 'True', 'S', etc.)
if 'srp' in df_25.columns:
    # Mostra os valores brutos que vieram da API antes de converter
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_25['srp'].value_counts(dropna=False))

    # Converte para booleano real
    df_25['srp_bool'] = df_25['srp'].astype(str).str.lower().isin(['true', '1', 's', 'sim'])

    df_25_com_srp = df_25[df_25['srp_bool'] == True]
    df_25_sem_srp = df_25[df_25['srp_bool'] == False]

    print(f"\n Total COM SRP: {len(df_25_com_srp)}")
    print(f" Total SEM SRP: {len(df_25_sem_srp)}")


Colunas encontradas relacionadas a SRP/Preço:
['srp']

Usando a coluna 'srp' como 'srp'

Valores brutos encontrados na coluna SRP:
srp
False    158
True      42
Name: count, dtype: int64

 Total COM SRP: 42
 Total SEM SRP: 158


In [35]:
print("\nDataFrame com SRP:")
display(df_25_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_25_sem_srp.head())


DataFrame com SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
5,94300105914952024,07954480000179-1-023841/2024,2024,23841,07954480000179,None,57870,ESTADO DO CEARA,None,E,...,138742.18,89250.00,2025-01-02T09:03:57,2025-01-02T09:03:57,2025-01-02T09:03:57,2025-01-03T08:00:00,2025-01-15T14:30:00,False,2025,True
7,94300105913952024,07954480000179-1-023842/2024,2024,23842,07954480000179,None,57870,ESTADO DO CEARA,None,E,...,759632.82,551284.00,2025-01-02T09:04:04,2025-01-02T09:04:04,2025-01-02T09:04:04,2025-01-03T08:00:00,2025-01-15T14:30:00,False,2025,True
8,94300105910162024,07954480000179-1-023843/2024,2024,23843,07954480000179,None,57870,ESTADO DO CEARA,None,E,...,55242.57,37068.54,2025-01-02T09:04:07,2025-01-02T09:04:07,2025-01-02T09:04:07,2025-01-03T08:00:00,2025-01-15T09:00:00,False,2025,True
10,98748705900602024,75972760000160-1-000185/2024,2024,185,75972760000160,None,84922,MUNICIPIO DE CAPANEMA,None,M,...,302156.25,206660.50,2025-01-02T09:04:14,2025-01-17T07:48:19,2025-01-02T09:04:14,2025-01-02T08:00:00,2025-01-21T08:30:00,False,2025,True
14,42512805900152024,02973240000106-1-000038/2024,2024,38,02973240000106,None,51122,ESTADO DO MARANHAO - SECRETARIA DE ESTADO DA S...,None,E,...,4425190.08,2782461.00,2025-01-02T09:04:26,2025-01-02T09:04:26,2025-01-02T09:04:26,2025-01-02T08:00:00,2025-01-15T09:00:00,False,2025,True



DataFrame sem SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
0,92647305900242024,06272868000127-1-000056/2024,2024,56,06272868000127,None,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,None,F,...,400000.00,NaN,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T09:03:34,2025-01-02T08:00:00,2025-01-16T09:00:00,False,2025,False
1,92647305900252024,06272868000127-1-000057/2024,2024,57,06272868000127,None,56302,CONSELHO REGIONAL DE ENFERMAGEM COREN MA,None,F,...,250782.10,151826.000,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T09:03:38,2025-01-02T08:00:00,2025-01-14T09:00:00,False,2025,False
2,38910305900162024,21947619000188-1-000068/2024,2024,68,21947619000188,None,21599,CONSELHO REGIONAL DE FISIOTERAPIA E TERAPIA OC...,None,F,...,868719.38,NaN,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:03:41,2025-01-02T09:00:00,2025-01-17T09:00:00,False,2025,False
3,94300105913862024,07954480000179-1-023839/2024,2024,23839,07954480000179,None,57870,ESTADO DO CEARA,None,E,...,11147211.58,6619438.535,2025-01-02T09:03:47,2025-05-09T08:13:51,2025-01-02T09:03:47,2025-04-25T08:00:00,2025-05-15T09:00:00,False,2025,False
4,38918505900112024,00119784000171-1-000036/2024,2024,36,00119784000171,None,43167,CONSELHO FEDERAL DE MEDICINA VETERINARIA,None,F,...,4612985.04,3718205.520,2025-01-02T09:03:51,2025-09-22T07:36:06,2025-01-02T09:03:51,2025-09-22T08:00:00,2025-10-07T10:00:00,False,2025,False


In [36]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_25 = df_25.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao_25.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao_25)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,2,81,158
Com SRP,1,30,42


# Comparação 2024 X 2025

In [37]:
display(df_qnt_modalidade_orgao_24)
display(df_qnt_modalidade_orgao_25)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,2,77,152
Com SRP,1,25,48


,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,2,81,158
Com SRP,1,30,42


In [38]:
df_total = pd.concat([df_24, df_25], ignore_index=True)
df_total

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,None,45178,MINISTERIO DA FAZENDA,None,F,...,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2024,True
1,41300105000262023,02030715000112-1-000226/2023,2023,226,02030715000112,None,89804,AGENCIA NACIONAL DE TELECOMUNICACOES,None,F,...,404.74,NaN,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T07:00:04,2024-01-02T08:00:00,2024-01-17T10:00:00,False,2024,False
2,15812605000492023,10729992000146-1-000127/2023,2023,127,10729992000146,None,7087,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",None,F,...,105000.00,105000.00,2024-01-02T07:00:05,2024-01-15T07:07:29,2024-01-02T07:00:05,2024-01-02T08:00:00,2024-01-16T10:00:00,False,2024,False
3,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,None,43849,COMANDO DA AERONAUTICA,None,F,...,286011.82,270000.00,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2024,False
4,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,None,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,None,F,...,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2024,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,92774706000032025,08907776000878-1-000001/2025,2025,1,08907776000878,None,3009,POLICIA MILITAR DO ESTADO - PM/PB,None,E,...,240975.00,240975.00,2025-01-02T18:46:26,2025-01-02T18:46:26,2025-01-02T18:46:26,NaN,NaN,False,2025,False
396,20010206000032025,26989715000102-1-000026/2025,2025,26,26989715000102,None,24182,MINISTERIO PUBLICO DA UNIAO,None,F,...,128822.00,128822.00,2025-01-02T19:01:13,2025-01-02T19:01:13,2025-01-02T19:01:13,NaN,NaN,False,2025,False
397,38908606000292024,15417520000171-1-000037/2024,2024,37,15417520000171,None,14636,CONSELHO REGIONAL DE ENGENHARIA E AGRONOMIA DE...,None,F,...,174154.35,172002.39,2025-01-02T19:03:12,2025-01-02T19:03:12,2025-01-02T19:03:12,NaN,NaN,False,2025,False
398,38927306930142025,06042030000147-1-000001/2025,2025,1,06042030000147,None,93943,CONSELHO REGIONAL DE SERVICO SOCIAL CRESS 2 RE...,None,F,...,900.00,887.00,2025-01-02T19:41:02,2025-01-02T19:41:02,2025-01-02T19:41:02,2025-01-02T19:41:01,2025-01-09T07:59:59,False,2025,False
